# Data Quality Analysis with Mistral 7B (4-bit Quantization)

This notebook demonstrates how to use the Mistral 7B language model (with 4-bit quantization) to analyze a sample dataset for data quality issues, using Hugging Face Transformers and the `bitsandbytes` library for efficient inference.

In [ ]:
# STEP 1: Install required libraries
!pip install torch torchvision torchaudio transformers accelerate bitsandbytes --quiet
!pip install -U bitsandbytes

In [ ]:
# STEP 2: Setup Hugging Face Token
import os

# Replace "your_token_here" with your Hugging Face access token
os.environ["HUGGINGFACE_TOKEN"] = "your_token_here"
from huggingface_hub import login
login(token=os.environ["HUGGINGFACE_TOKEN"])

In [ ]:
# STEP 3: Load Mistral 7B with 4-bit quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

quant_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ["HUGGINGFACE_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, token=os.environ["HUGGINGFACE_TOKEN"])

In [ ]:
import pandas as pd

# Sample dataset
data = {
    'CustomerID': [1001, 1002, 1003, None, 1005],
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eve'],
    'Email': ['alice@gmail', 'bob@yahoo.com', 'charlie@@mail.com', None, 'eve@gmail.com'],
    'JoinDate': ['2021-01-01', '2021-02-30', '2021-03-15', 'bad_date', '2021/04/01'],
    'Country': ['MY', 'Malaysia', 'MY', 'Singapore', None],
    'Revenue': ['1000', 'Two Thousand', '3000', '-500', '4000']
}
df = pd.DataFrame(data)

In [ ]:
# Convert schema to prompt
schema = str(df.dtypes)
prompt = f"""
You are a data quality expert. Analyze the schema below and:
1. Identify 3 potential data quality issues.
2. Suggest data profiling steps.
3. Recommend data standardization or validation actions.

Schema:
{schema}
"""

In [ ]:
# Tokenize input
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate response
outputs = model.generate(**inputs, max_new_tokens=300)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print only response (remove prompt from output)
print(response[len(prompt):].strip())

## What does this code do? (ELI5)

This notebook shows how to use a powerful AI model (Mistral 7B) to check a small dataset for data quality problems. Here’s what happens step by step:

1. **Install libraries:** It installs all the tools needed to run the AI model.
2. **Set up access:** You add your Hugging Face token so you can use the model.
3. **Load the model:** It loads the Mistral 7B model in a memory-efficient way (4-bit quantization).
4. **Create sample data:** It makes a small table with some fake customer data, including some errors.
5. **Build a prompt:** It asks the AI to look at the data’s structure and find possible issues, suggest ways to profile the data, and recommend how to clean it up.
6. **Run the model:** The AI reads the prompt and gives advice on data quality.

**In short:** This code uses a big AI model to act like a data quality expert, helping you spot and fix problems in your data!